# 05 - Gap-edge residual correction models

This notebook introduces **gap-edge residual correction models**: a model
family that predicts a correction *on top of* linear interpolation, using
information from both edges of a gap, rather than predicting the missing
value directly from scratch.

It covers:

1. what a gap-edge residual correction model is;
2. why it is fundamentally retrospective, not forecast-safe;
3. how it predicts a correction over linear interpolation;
4. what features it sees;
5. when it helps and when it fails;
6. how it compares, as a classical machine-learning comparator, to the
   zero-shot TS-ICL foundation model;
7. an illustration using the public benchmark tables and figures (full
   retraining is out of scope for this notebook).

## 1. What is a gap-edge residual correction model?

Linear interpolation between the last observation before a gap and the
first observation after it is already a strong baseline for short,
smooth gaps (see `03_baselines.ipynb`). A gap-edge residual correction
model starts from that same linear interpolation and learns to predict the
**residual** -- how far the true (withheld) value is likely to deviate
from the straight-line interpolation -- using extra information available
at both edges of the gap and from engineered external predictors (see
`04_engineered_tabular_models.ipynb`).

This is a different strategy from training a model to predict the target
value directly: by construction, the model only has to learn the
*correction*, which can be a smaller, easier-to-learn signal than the raw
target value, especially when interpolation is already a reasonable
starting point.

## 2. Why this approach is retrospective, not forecast-safe

A gap-edge residual model needs an observation **after** the gap (the
"post-edge" value) to compute the interpolation baseline it corrects, and
to construct several of its own input features (e.g. a post-gap rolling
summary). That means it can only be used:

- to fill an **artificial gap** in a validation experiment, where the
  surrounding real data exists and was only masked for testing, or
- to retrospectively reconstruct a **real gap** after the sensor came
  back online and post-gap data became available.

It **cannot** be used for live forecasting (predicting today's value
without knowing tomorrow's), because the post-edge information it depends
on does not exist yet in a forecasting setting. This mirrors the same
caveat already noted for plain linear interpolation in
`docs/methodology/model_families.md`, and is why this project's scope is
diagnostic reconstruction, not forecasting (see the project's top-level
scope notes).

## 3. How the correction is predicted

At a high level:

1. Compute the linear interpolation baseline for the gap (using the
   pre-edge and post-edge observed values).
2. Compute a set of gap-edge and engineered-predictor features for the
   gap (see section 4).
3. Train a regression model (e.g. ridge regression, random forest,
   gradient boosting) to predict `true_value - interpolation_baseline`
   (the residual) from those features, using gaps where the true value is
   known (i.e. artificial gaps in the validation pool).
4. At prediction time, for a new gap, compute the interpolation baseline
   and the same features, predict the residual, and add it back:
   `prediction = interpolation_baseline + predicted_residual`.

This keeps the model's job bounded: even if the residual prediction is
poor, the result degrades gracefully toward the interpolation baseline
rather than producing an arbitrary value.

## 4. What features the model sees

- **Gap-edge values** -- the last observed value before the gap and the
  first observed value after it (and possibly their difference, or
  short rolling summaries computed only from observed days adjacent to
  the gap).
- **Pre/post summaries** -- longer trailing/leading windows around the
  gap (e.g. a 7-day mean before the gap, a 7-day mean after it), which can
  indicate whether the surrounding period was unusually high or low.
- **Engineered external predictors** -- the same family of calendar,
  SST, wind, and upwelling features described in
  `04_engineered_tabular_models.ipynb`, evaluated during the gap window,
  which can indicate whether physical forcing during the gap looked
  different from the smooth interpolation would assume.
- **Gap length** -- since the reliability of interpolation (and the
  amount of new information a correction can safely add) changes with how
  long the gap is.

## 5. When it helps, and when it fails

**Helps**: longer or smoother gaps, where linear interpolation's
straight-line assumption is least accurate but the surrounding context
(pre/post edges, external forcing) is informative -- the correction model
can recover curvature or a level shift that a straight line misses.

**Fails**: gaps that hide a real event -- a bloom peak or other sharp,
short-lived spike in chlorophyll that neither the gap edges nor the
external predictors clearly signal in advance. Every method in this
benchmark, including the leading TS-ICL configuration, systematically
under-predicts high-chlorophyll event days (see
`docs/methodology/event_limitation.md` and
`results_public/chlorophyll/chlorophyll_event_performance_summary.csv`);
a gap-edge residual model is no exception, since a hidden, short-lived
spike rarely leaves a strong enough trace in the edge values or external
forcing to be predicted reliably.

## 6. A classical comparator to the zero-shot foundation model

This model family -- along with the Gaussian process and state-space
components described in `docs/methodology/model_families.md` -- forms the
"engineered hybrid pipeline" released as
`results_public/chlorophyll/chlorophyll_reconstruction_engineered_hybrid.csv`.
It represents a carefully engineered, classical machine-learning approach:
deliberately designed features, a validation-aware policy for choosing
which method family to trust at each gap length, and full transparency
about how each prediction was produced.

TS-ICL (`06_tsicl_zero_shot_imputation.ipynb`) takes the opposite
approach: a large pretrained time-series foundation model applied with no
site-specific fine-tuning, given the gappy series (optionally with a
satellite chlorophyll proxy covariate) and asked to fill the gap directly.

Under this project's validation-grade testing
(`results_public/chlorophyll/chlorophyll_benchmark_summary.csv`), the
TS-ICL satellite-proxy configuration shows a statistically significant
improvement in mean absolute error over the engineered hybrid pipeline,
across most gap lengths. The engineered hybrid pipeline remains a useful,
fully transparent comparator -- and a reasonable choice when a
fully-explainable, no-external-dependency pipeline is preferred over a
foundation model -- but the validation evidence currently favors the
zero-shot foundation model approach in this setting.

## 7. Illustration using public result tables and figures

Full retraining of the gap-edge residual component is out of scope for
this notebook (it is one piece of the larger engineered hybrid pipeline,
whose full private training code is not republished here -- see
`docs/methodology/model_families.md`). Instead, this section loads the
already-computed public benchmark tables and figure to show how the
engineered hybrid pipeline compares to TS-ICL and the simpler baselines
under validation-grade testing.

In [ ]:
import pandas as pd

benchmark = pd.read_csv(
    "../results_public/chlorophyll/chlorophyll_benchmark_summary.csv"
)
gap_scores = pd.read_csv(
    "../results_public/chlorophyll/chlorophyll_artificial_gap_scores.csv"
)

# Pairwise comparisons involving the engineered hybrid pipeline, all gaps.
hybrid_name = "Engineered hybrid pipeline (validation-aware method assignment)"
hybrid_rows = benchmark[
    (benchmark["method_public_name"] == hybrid_name)
    | (benchmark["compared_against_public_name"] == hybrid_name)
]
hybrid_rows[[
    "method_public_name", "compared_against_public_name", "stratum",
    "n_gaps", "method_value", "comparison_value", "delta", "interpretation",
]]

In [ ]:
# Error by gap length for the engineered hybrid pipeline vs. TS-ICL and
# linear interpolation, using the long-format scores table.
methods_of_interest = [
    hybrid_name,
    "TS-ICL (satellite chlorophyll proxy covariate)",
    "Linear interpolation baseline",
]
by_length = gap_scores[
    (gap_scores["stratum_type"] == "gap_length")
    & (gap_scores["method_public_name"].isin(methods_of_interest))
]
pivot = by_length.pivot_table(
    index="stratum_value", columns="method_public_name", values="day_weighted_mae",
    aggfunc="mean",
)
pivot

In [ ]:
from PIL import Image

# The gap-length performance figure visualizes this
# length-by-length comparison across all evaluated methods.
img = Image.open("../figures/chlorophyll/figure_gap_length_performance.png")
img

## 8. Takeaways

- A gap-edge residual correction model predicts a correction over linear
  interpolation, using both gap edges, pre/post summaries, and engineered
  external predictors.
- It is retrospective by construction (it needs a post-gap observation),
  so it is only usable for artificial-gap validation or after-the-fact
  reconstruction of real gaps -- never for live forecasting.
- It tends to help most on longer/smoother gaps and tends to fail on
  hidden events (sharp, short-lived chlorophyll spikes), the same
  limitation that affects every method in this benchmark.
- As a classical machine-learning comparator to the zero-shot TS-ICL
  foundation model, it is fully transparent and dependency-light, but the
  validation-grade evidence in this benchmark currently favors TS-ICL with
  a satellite chlorophyll proxy covariate -- see
  `results_public/chlorophyll/chlorophyll_benchmark_summary.csv` and
  `docs/evidence_hierarchy.md` before drawing any stronger conclusion.